# Node 04a · 受体制备（6W63 / 3V3M / 4YLU）· v0.2.3
### 清洗 → 几何/口袋/尺寸断言 → Meeko 生成受体 PDBQT

调度 `core/node04a_receptor_prep.py` v0.2.3。已在真实结构上端到端验证：
- **统一清洗**：三受体同一流程 → 链A蛋白重原子、去配体/溶剂/离子/水/altlocB/H。
- **口袋清空断言** + **盒中心自动**（X77/0EN/R30 质心）+ **盒尺寸核验**（27 Å 容纳 23.2 Å 最大配体）。
- **Meeko 转换**（已确认工具，ADFR 未装）：`mk_prepare_receptor.py --read_pdb {in} -o {out_base} -p -a`
  - `-o {out_base}`：Meeko 的 `-o` 需 basename，`-p` 写 `<basename>.pdbqt`。
  - `-a`：丢弃缺原子残基（如 3V3M 的 C 端 GLN306，远离活性位点）而非报错。
  - **质子化**：Meeko 按残基模板加极性 H（标准态；催化二联体为中性 His+中性 Cys）。
  - **v0.2.3 安全断言**：被丢弃的残基【绝不能】含催化二联体，否则报错。

**已验证结果**：三者均 口袋清空✅ / 已质子化（494–525 极性H）/ 二联体完整；仅 3V3M 丢弃 A:306。
**顺序**：Cell 1→2→3(清洗)→4→5(几何核验)→6(转换)。DRY_RUN 只把关 Cell 6 的写盘。

In [1]:
# ── Cell 0 · PATH（JupyterHub 未把 md_env/bin 置入 PATH）──
import os, sys
from pathlib import Path
_ENV_BIN = str(Path(sys.executable).parent)
if _ENV_BIN not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = _ENV_BIN + os.pathsep + os.environ["PATH"]
print("PATH[0] =", os.environ["PATH"].split(os.pathsep)[0])

PATH[0] = /mnt/hpc/home/wanghongmei/anaconda3/envs/md_env/bin


In [2]:
# ── Cell 1 · 配置 ──────────────────────────────────────
from pathlib import Path
import sys
DRY_RUN = False
PROJECT_ROOT = Path("/mnt/hpc/home/wanghongmei/lang2").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
NODE02_SDF       = PROJECT_ROOT / "results" / "02_master_molecules" / "sdf"
RAW_RECEPTOR_DIR = PROJECT_ROOT / "data" / "receptors"
OUT_DIR          = PROJECT_ROOT / "results" / "04_docking" / "receptors"

PDB_PATHS = {
    "6W63": RAW_RECEPTOR_DIR / "6W63.pdb",
    "3V3M": RAW_RECEPTOR_DIR / "3V3M.pdb",
    "4YLU": RAW_RECEPTOR_DIR / "4YLU.pdb",
}
COCRYSTAL_LIGAND_3V3M = "0EN"
COCRYSTAL_LIGAND_4YLU = "R30"

# 受体制备工具（已确认 Meeko 可用、ADFR 未装；命令已在真实结构验证）：
TOOL_CMD_TEMPLATE = "mk_prepare_receptor.py --read_pdb {in} -o {out_base} -p -a"
ALLOWED_HET = set()
BOX_SIZE = (27.0, 27.0, 27.0)

print("DRY_RUN =", DRY_RUN)
for k,p in PDB_PATHS.items(): print(f"  raw {k}: {p} 存在:{p.is_file()}")

DRY_RUN = False
  raw 6W63: /mnt/hpc/home/wanghongmei/lang2/data/receptors/6W63.pdb 存在:True
  raw 3V3M: /mnt/hpc/home/wanghongmei/lang2/data/receptors/3V3M.pdb 存在:True
  raw 4YLU: /mnt/hpc/home/wanghongmei/lang2/data/receptors/4YLU.pdb 存在:True


In [3]:
# ── Cell 2 · 导入 + 应用配置 ───────────────────────────
import importlib, json, copy
from core import node04a_receptor_prep as RP
importlib.reload(RP)
print("受体制备模块:", RP.RECEPTOR_PREP_VERSION)
RECEPTORS = copy.deepcopy(RP.RECEPTORS)
RECEPTORS["3V3M"]["cocrystal_ligand"] = COCRYSTAL_LIGAND_3V3M
RECEPTORS["4YLU"]["cocrystal_ligand"] = COCRYSTAL_LIGAND_4YLU
for k in RECEPTORS: RECEPTORS[k]["box_size"] = tuple(BOX_SIZE)
for k,c in RECEPTORS.items():
    dyad="/".join(f"{n}{s}" for n,s in c["catalytic_dyad"])
    print(f"  {k} ({c['virus']}): 二联体 {dyad} | 盒中心 自动(HET={c['cocrystal_ligand']}) | 盒 {c['box_size'][0]} Å")

受体制备模块: 0.2.6
  6W63 (SARS-CoV-2): 二联体 HIS41/CYS145 | 盒中心 自动(HET=X77) | 盒 27.0 Å
  3V3M (SARS-CoV-1): 二联体 HIS41/CYS145 | 盒中心 自动(HET=0EN) | 盒 27.0 Å
  4YLU (MERS-CoV): 二联体 HIS41/CYS148 | 盒中心 自动(HET=R30) | 盒 27.0 Å


In [4]:
# ── Cell 3 · 统一受体清洗（生成 protein-only）──
CLEAN_PDBS = {}
for pid in ("6W63","3V3M","4YLU"):
    outp = RAW_RECEPTOR_DIR / f"{pid}_protein_only.pdb"
    a = RP.clean_receptor_pdb(PDB_PATHS[pid], outp, chain="A")
    chk = RP.check_clean_receptor(outp, allowed_het=ALLOWED_HET)
    CLEAN_PDBS[pid] = outp
    ok = chk["n_remaining_non_water_het"]==0 and chk["n_water_atoms_present"]==0
    print(f"  {pid}: kept {a['n_atoms_kept']} | 非水HET={chk['remaining_non_water_het']} 水={chk['n_water_atoms_present']} "
          + ("✅ 纯蛋白" if ok else "❌"))

  6W63: kept 2358 | 非水HET=[] 水=0 ✅ 纯蛋白
  3V3M: kept 2368 | 非水HET=[] 水=0 ✅ 纯蛋白
  4YLU: kept 2312 | 非水HET=[] 水=0 ✅ 纯蛋白


In [5]:
# ── Cell 4 · 实测候选池最大配体长轴（盒尺寸依据）──
from rdkit import Chem
from rdkit import RDLogger; RDLogger.DisableLog("rdApp.*")
def longest_axis(mol):
    conf=mol.GetConformer(); P=[conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())]
    hv=[i for i in range(mol.GetNumAtoms()) if mol.GetAtomWithIdx(i).GetAtomicNum()>1]; m=0.0
    for a in range(len(hv)):
        for b in range(a+1,len(hv)):
            i,j=hv[a],hv[b]; d=((P[i].x-P[j].x)**2+(P[i].y-P[j].y)**2+(P[i].z-P[j].z)**2)**0.5
            if d>m: m=d
    return m
axes=[]
for sdf in sorted(NODE02_SDF.glob("*.sdf")):
    mol=Chem.MolFromMolFile(str(sdf), removeHs=False)
    if mol and mol.GetNumConformers(): axes.append((sdf.stem, longest_axis(mol)))
axes.sort(key=lambda x:-x[1]); POOL_MAX_AXIS=axes[0][1]
print(f"候选池 n={len(axes)} | 最大长轴 {POOL_MAX_AXIS:.1f} Å ({axes[0][0]}) | 盒边 {min(BOX_SIZE)} Å → 每侧余量 {(min(BOX_SIZE)-POOL_MAX_AXIS)/2:.1f} Å")

候选池 n=1765 | 最大长轴 23.2 Å (pool_mol_756_s01) | 盒边 27.0 Å → 每侧余量 1.9 Å


In [6]:
# ── Cell 5 · 几何 + 盒尺寸核验（dry）──────────────────
print(f"{'受体':<6}{'二联体':<15}{'盒中心':<30}{'→二联体(Å)':<11}{'盒余量(Å)':<10}结果")
print("-"*88)
all_ok=True
for pid,pth in PDB_PATHS.items():
    cfg=RECEPTORS[pid]
    try:
        m=RP.prepare_receptor(pid, pth, OUT_DIR, receptors=RECEPTORS, pool_max_axis=POOL_MAX_AXIS, dry_run=True)
        dyad="/".join(str(x) for x in m["catalytic_dyad"]); ctr="("+", ".join(f"{c:.2f}" for c in m["box_center"])+")"
        d=m["box_sanity"]["center_to_dyad_angstrom"]; mar=m["box_fit"]["margin_per_side_angstrom"]
        print(f"{pid:<6}{dyad:<15}{ctr:<30}{d:<11}{mar:<10}✅ {m['box_center_source']}")
        if m["box_fit"].get("warning"): print(f"       ⚠ {m['box_fit']['warning'][:60]}")
    except Exception as e:
        all_ok=False; print(f"{pid:<6}{'':<15}{'':<30}{'':<11}{'':<10}❌ {str(e)[:55]}")
print("-"*88); print("✅ 核验通过。" if all_ok else "❌ 有受体未过。")

受体    二联体            盒中心                           →二联体(Å)    盒余量(Å)    结果
----------------------------------------------------------------------------------------
6W63  HIS41/CYS145   (-20.10, 18.84, -27.39)       4.753      1.88      ✅ auto_from_cocrystal_ligand
3V3M  HIS41/CYS145   (23.55, -27.25, -2.30)        4.578      1.88      ✅ auto_from_cocrystal_ligand
4YLU  HIS41/CYS148   (37.48, -18.46, 37.49)        4.515      1.88      ✅ auto_from_cocrystal_ligand
----------------------------------------------------------------------------------------
✅ 核验通过。


In [7]:
# ── Cell 5b · 工具可用性诊断 ──
import shutil, sys, os
from pathlib import Path
print("kernel python :", sys.executable)
print("conda env     :", os.environ.get("CONDA_DEFAULT_ENV"))
print("which()       :", shutil.which("mk_prepare_receptor.py"))
sib = Path(sys.executable).parent / "mk_prepare_receptor.py"
print("sibling bin   :", sib, sib.is_file())
try:
    import meeko; print("meeko module  :", meeko.__version__)
except ImportError as e:
    print("meeko module  : NOT importable ->", e)

kernel python : /mnt/hpc/home/wanghongmei/anaconda3/envs/md_env/bin/python
conda env     : jupyterhub
which()       : /mnt/hpc/home/wanghongmei/anaconda3/envs/md_env/bin/mk_prepare_receptor.py
sibling bin   : /mnt/hpc/home/wanghongmei/anaconda3/envs/md_env/bin/mk_prepare_receptor.py True
meeko module  : 0.7.1


In [8]:
# ── Cell 6 · 口袋清空断言 + Meeko 生成受体 PDBQT + manifest ──
manifest = RP.run_receptor_prep(
    PDB_PATHS, OUT_DIR, receptors=RECEPTORS,
    tool_cmd_template=TOOL_CMD_TEMPLATE, clean_pdbs=CLEAN_PDBS,
    allowed_het=ALLOWED_HET, pool_max_axis=POOL_MAX_AXIS, dry_run=DRY_RUN)
for k,m in manifest["receptors"].items():
    cc=m.get("clean_receptor_check"); c=m.get("conversion")
    print(f"  {k}: 盒{m['box_center']} | 二联体{m['catalytic_dyad']}"
          f" | 口袋清空:{'✅' if (cc and cc['n_remaining_non_water_het']==0) else '❌'}"
          + (f" | 转换✅ {c['n_pdbqt_atoms']}原子 {c['n_polar_h']}极性H 丢弃{c['dropped_incomplete_residues'] or '无'}"
             if (c and c.get('executed')) else " | 转换:(dry/未配)"))
if not DRY_RUN and TOOL_CMD_TEMPLATE:
    print(f"\n✓ manifest: {OUT_DIR}/Node04a_receptor_manifest.json")
    print("✅ 受体制备冻结。下一步：Node 04 对接（5 种子 / exh 32 / modes 9）。")

  6W63: 盒[-20.096, 18.844, -27.394] | 二联体['HIS41', 'CYS145'] | 口袋清空:✅ | 转换✅ 2879原子 521极性H 丢弃无
  3V3M: 盒[23.548, -27.252, -2.297] | 二联体['HIS41', 'CYS145'] | 口袋清空:✅ | 转换✅ 2886原子 525极性H 丢弃['A:306']
  4YLU: 盒[37.484, -18.463, 37.49] | 二联体['HIS41', 'CYS148'] | 口袋清空:✅ | 转换✅ 2806原子 494极性H 丢弃无

✓ manifest: /mnt/hpc/home/wanghongmei/lang2/results/04_docking/receptors/Node04a_receptor_manifest.json
✅ 受体制备冻结。下一步：Node 04 对接（5 种子 / exh 32 / modes 9）。


In [9]:
# ── Cell 7 · manifest 预览 ─────────────────────────────
import json
print(json.dumps(manifest, indent=2, ensure_ascii=False)[:2200])

{
  "module": "node04a_receptor_prep",
  "version": "0.2.6",
  "created_at_utc": "2026-07-28T14:20:26Z",
  "dry_run": false,
  "max_center_to_dyad_angstrom": 8.0,
  "pool_max_ligand_axis_angstrom": 23.248200774683614,
  "dyad_protonation_uniform": {
    "6W63": {
      "HIS": "HIE_epsilon",
      "CYS": "SH_neutral_thiol"
    },
    "3V3M": {
      "HIS": "HIE_epsilon",
      "CYS": "SH_neutral_thiol"
    },
    "4YLU": {
      "HIS": "HIE_epsilon",
      "CYS": "SH_neutral_thiol"
    }
  },
  "policy": {
    "catalytic_dyad": "per-ortholog; MERS(4YLU)=His41/Cys148 (corrected from Cys145)",
    "box_center": "auto from co-crystal ligand centroid where present; else fixed_from_verified_reference (traceable); no bare hardcode",
    "box_center_sanity": "asserted <= 8.0 A from dyad midpoint (rejects ~29.82 A fallback)",
    "clean_receptor": "asserted NO non-water HETATM in cleaned receptor (co-crystal ligand must be removed; else occupied-pocket silent error)",
    "box_size": "asserted 

In [1]:
import sys, json
R = "/mnt/hpc/home/wanghongmei/lang2"
sys.path.insert(0, R)
from core import node04_docking as N4

gate, recs, m4a = N4.verify_upstream(
    f"{R}/results/03_ligand_pdbqt/Node03_manifest.json",
    f"{R}/results/03_ligand_pdbqt/identity_manifests_node03.jsonl",
    f"{R}/results/04_docking/receptors/Node04a_receptor_manifest.json")
print(json.dumps(gate, indent=2, ensure_ascii=False))

pc, ch, st = N4.derive_pool_class(f"{R}/results/02_master_molecules/identity_manifests.jsonl")
print("pool_class:", st)

{
  "node03_manifest_sha256": "3fc55501ae273b1885eac3cb7386acf6798eea53e94f3f3b8eb5aff5184412c3",
  "node03_jsonl_sha256": "2455a65683904468c310058096f6cca31cbab759cd67c66475c9791b3a1131d4",
  "node04a_manifest_sha256": "7c6dd6404f7a6ed0dfa7309c5ff16dc696ebdb23e60e0413320c2cbd05f0b814",
  "n_ligands_verified": 1765,
  "n_receptors_verified": 3,
  "node03_version": "0.2.0",
  "node04a_version": "0.2.6",
  "assertion": "all upstream hashes re-derived and matched"
}
pool_class: {'heavy_atom_min': 12, 'n_fragment': 76, 'n_candidate': 1689, 'primary_source': 'node02 frozen heavy_atom_count', 'cross_check_vs_node00': None}
